### Library importations 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
from matplotlib import pyplot as plt
import pandas as pd
import matplotlib
matplotlib.use('QtAgg') 


### Global variables

In [ ]:
baseline_window = 10
window_test = 30 # sliding window over signal 
window = 50 
threshold_test = 100 
r0 = 3 
step = 1
frq = 250

## Initializing Scoring Results

### Plotting functions 

In [2]:
def on_key(event):
    global current_index, fig, subject_epoch, subject, block,window_size

    if event.key == 'right':
        current_index = (current_index + 1) % len(subject_epoch)
    elif event.key == 'left':
        current_index = (current_index - 1) % len(subject_epoch)
    elif event.key == 'escape':
        print("Quitting the plot!")
        plt.close(fig)
        return

    epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
    epoch_zygo = epoch_zygo*10000

    epoch_var_zygo, _, _, _ = get_features(epoch_zygo)


    plt.close(fig)
    fig = plot_test_variance_window_func(epoch_zygo,epoch_var_zygo)
  
    fig.canvas.mpl_connect('key_press_event', on_key)   
    # fig.suptitle(f"subject: {subject} | epoch: {current_index}", fontsize=14)


    plt.show(block=False)


In [3]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples
def get_features(epoch):
     global frq
     global window
     global step

     # pad epoch to preserve sample number
     pad_left  = window // 2
     pad_right = window - 1 - pad_left  
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

     # frequency features (did not use)
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]

 
     return var, rms, wl, fmd

In [ ]:
def plot_test_variance_window_func(zygo,feature_zygo):
    global current_index
    fig, ax = plt.subplots(1, 1, figsize=(8, 4), sharex=True)
   


    ax_zygo = ax.twinx()

    zygomatic_color = "#6F9359"

    time = np.linspace(0,7,len(zygo))
    trigger_time = time[0]

    zygo_out,thr_zyg = test_variance_window_func(feature_zygo)
    zygo_out = zygo_out.astype(int)

    lengths, starts, ends = extract_periods(zygo_out)
    cluster_flg,flagged_idx_c = flag_clusters(zygo_out,30)
    period_flg,flagged_idx_p = flag_period(zygo_out,.5)

    if (period_flg or cluster_flg):
        if (period_flg):
            title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {lengths} | INCORRECT LABELS",color="red")
            for i in flagged_idx_p:
                ax.axvspan(starts[i] / frq,
                ends[i] / frq,
                color="grey",
                alpha=0.3)
        if (cluster_flg):
            title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {lengths} | INCORRECT LABELS",color="red")
            for i in flagged_idx_c:
                ax.axvspan(starts[i] / frq,
                ends[i] / frq,
                color="red",
                alpha=0.3)
    else: 
        title = fig.suptitle(f"Epoch: {current_index} | DETECTED: {lengths} | OK LABELS",color="gold")


                    
    # Plot Zygo
    ax.plot(time, zygo, lw=1.5, color=zygomatic_color)
   # ax[0].axhline(thr_zyg,linestyle='--')
    # ax_zygo.plot(time,feature_zygo)
    ax.set_ylim(-200, 200)
    ax.set_ylabel("Zygo",size=12)
    # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))




    ax_zygo.plot(time,zygo_out,  color="red", label="computed labels", alpha=0.7,)

    ax_zygo.axis("on")

    plt.legend() 

    return fig 

### Processing functions 

In [6]:
def test_variance_window_func(feature):
    global window_test,baseline_window,threshold_test, window 

    events = np.zeros(len(feature),dtype=bool)

    baseline_mean_start = np.mean(feature[:baseline_window])
    baseline_mean_end = np.mean(feature[-baseline_window:-1])

    #----------NO RESPONSE CASE LOGIC--------------
    if ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.max(feature))) < 250) or ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.mean(feature))) < 10):
        return events, 0 
    
    #----------RESPONSE CASE LOGIC--------------
    max_var = 0 
    for start in range(0, len(feature) - window + 1, 1):
        stop = start + window
        feature_win = feature[start:stop]
        curr_var = np.mean(feature_win)

        if (curr_var >  max_var): 
            max_var = curr_var 
    

    thr =   max_var/4
    ind = np.where((feature> thr))
    events[ind] = True 

    
    return events,thr


In [7]:

''' 
def set_second_threshold(events):
    global r0 

    for i in range(len(events),len(events)-r0)
'''

' \ndef set_second_threshold(events):\n    global r0 \n\n    for i in range(len(events),len(events)-r0)\n'

In [8]:
path = '/Users/basak/Desktop/Research/Projects/INCC/SoundSleep/results/pilots/Cami_sleep.vhdr'
path = '/Users/basak/Desktop/Research/Projects/INCC/SoundSleep/results/pilots/S02_sleep_active.vhdr'


subject = 'Cami'#'Cami' #"S01"
current_index = 0 

### Flagging Functions

In [33]:
def extract_periods(labels):
    padded = np.pad(labels, (1, 1), constant_values=0)
    diff = np.diff(padded)

    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]

    lengths = ends - starts
    return lengths, starts, ends 

In [ ]:
# flags if the period between contractions is longer 
# set max period to be five seconds 
def flag_period(labels,max_period=5):
    lengths,_,_ = extract_periods(labels)
    periods = lengths/frq

    flagged_idx = np.where(periods > max_period)[0]
    print(periods)
    if np.any(periods > max_period):
        return True,flagged_idx
    else:
        return False, None
    

In [53]:
# flags if the length of clusters is too short 
# set min period to be 100 sampels 
def flag_clusters(labels,min_len=100):
    lengths, _, _ = extract_periods(labels)
    flagged_idx = np.where(lengths < min_len)[0]

    if np.any(lengths < min_len):
        return True, flagged_idx
    else:
        return False, None


### Pre-processing 

In [10]:
raw = mne.io.read_raw_brainvision('/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/Cami_sleep.vhdr', preload=True)
# raw = mne.io.read_raw_brainvision('/Users/basak/Desktop/Research/Projects/INCC/SoundSleep/results/pilots/S02_sleep_active.vhdr', preload=True)


emg_ch= ['Zygo', 'Menton']


if subject=='Cami':  # For Cami, EOG was recorded on IO channel
    mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo'})
    raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog'})
    mne.add_reference_channels(raw, ref_channels=['C3'], copy=False)
else:
    mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo','69':'EOG'})
    raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog','EOG':'eog'})  
    mne.add_reference_channels(raw, ref_channels=['Cz'], copy=False)  
if subject=='S02':
    idx=raw.ch_names.index('EOG')
    raw._data[idx]*=-1

#raw = mne.set_bipolar_reference(raw, anode='IO', cathode='AF7',drop_refs=False) #
raw.resample(sfreq=250)

#raw.set_eeg_reference(['TP10'], projection=False) 

emg_filter_params = {'lpass': 100,'hpass': 10,'notches': [50]}
eeg_eog_filter_params = {'lpass': 15,'hpass': 0.3,'notches': [50]}
ecg_filter_params = {'lpass': 70,'hpass': 0.3,'notches': [50]}

raw.filter(l_freq=emg_filter_params['hpass'],h_freq=emg_filter_params['lpass'],picks=emg_ch)
#raw.filter(l_freq=eeg_eog_filter_params['hpass'],h_freq=eeg_eog_filter_params['lpass'],picks=mne.pick_types(raw.info, eeg=True, eog=True))
#raw.filter(l_freq=ecg_filter_params['hpass'],h_freq=ecg_filter_params['lpass'],picks=mne.pick_types(raw.info, ecg=True))

events_all= mne.events_from_annotations(raw)[0]
events= mne.pick_events(events_all,include=list(range(1,40))+[99]) # Pick events of interest (where a stimulus was presented)

df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
df_triggers.drop(columns=['dunno'],inplace=True)
df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']



epochs = mne.Epochs(raw, events, tmin=-0, tmax=7,baseline=None, detrend=0,
                reject=None, preload=True, on_missing='warn')

subject_epoch = epochs 

Extracting parameters from /Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/Cami_sleep.vhdr...
Setting channel info structure...
Reading 0 ... 15375049  =      0.000 ...  6150.020 secs...


/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_49876/2530392507.py:1: RuntimeWarning: No coordinate information found for channels ['IO', '65', '66', '67', '68', '69', '70', '71', '72']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision('/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/Cami_sleep.vhdr', preload=True)
/var/folders/vy/grssbq395j340gpb616mgthm0000gn/T/ipykernel_49876/2530392507.py:1: RuntimeWarning: Not setting positions of 9 misc channels found in montage:
['IO', '65', '66', '67', '68', '69', '70', '71', '72']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision('/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/Cami/Cami_sleep.vhdr', preload=True)
/var/folde

Filtering a subset of channels. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 10 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 10.00
- Lower transition bandwidth: 2.50 Hz (-6 dB cutoff frequency: 8.75 Hz)
- Upper passband edge: 100.00 Hz
- Upper transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 112.50 Hz)
- Filter length: 331 samples (1.324 s)

Used Annotations descriptions: ['Stimulus/S  1', 'Stimulus/S  2', 'Stimulus/S  3', 'Stimulus/S  4', 'Stimulus/S  5', 'Stimulus/S  6', 'Stimulus/S  7', 'Stimulus/S  8', 'Stimulus/S  9', 'Stimulus/S 11', 'Stimulus/S 12', 'Stimulus/S 13', 'Stimulus/S 14', 'Stimulus/S 15', 'Stimulus/S 16', 'Stimulus/S 17', 'Stimulus/S 18', 'S

## Scoring GUI

In [106]:
# extract epoch  
current_index = 253
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_zygo = epoch_zygo*10000

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)


fig = plot_test_variance_window_func(epoch_zygo,epoch_var_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


plt.show()

[0.212 0.352 0.304]


KeyboardInterrupt: 